In [5]:
# Install dependencies
!pip install timm diffusers


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Download ImageNet Validation Dataset (ILSVRC2012)
# This will download ~6.3GB and organize it into class folders

import os
import tarfile
from pathlib import Path
import xml.etree.ElementTree as ET

# Download validation set
print('Downloading ImageNet validation set (~6.3GB)...')
!wget -nc https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar

# Download validation ground truth annotations
print('Downloading validation ground truth...')
!wget -nc https://image-net.org/data/ILSVRC/2012/ILSVRC2012_devkit_t12.tar.gz

# Extract validation images
print('Extracting validation images...')
os.makedirs('imagenet_val', exist_ok=True)
!tar -xf ILSVRC2012_img_val.tar -C imagenet_val

# Extract devkit to get ground truth
print('Extracting devkit...')
!tar -xzf ILSVRC2012_devkit_t12.tar.gz

# Parse ground truth from devkit
print('Parsing validation ground truth...')
gt_file = 'ILSVRC2012_devkit_t12/data/ILSVRC2012_validation_ground_truth.txt'
with open(gt_file, 'r') as f:
    # Ground truth file has 1-indexed class IDs, convert to 0-indexed
    labels = [int(line.strip()) - 1 for line in f]

# Organize into class folders
print('Organizing into class folders...')
val_dir = Path('imagenet_val')
organized_dir = Path('imagenet_val_organized')

# Create class directories
for class_id in set(labels):
    (organized_dir / str(class_id)).mkdir(parents=True, exist_ok=True)

# Move images to class folders
val_images = sorted(val_dir.glob('ILSVRC2012_val_*.JPEG'))
for idx, img_path in enumerate(val_images):
    class_id = labels[idx]
    target_path = organized_dir / str(class_id) / img_path.name
    img_path.rename(target_path)

print(f'✓ ImageNet validation set organized into {len(set(labels))} class folders')
print(f'Total images: {len(val_images)}')
print('Dataset ready at: ./imagenet_val_organized')


--2026-05-27 19:55:09--  https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar
Resolving image-net.org (image-net.org)... 171.64.68.16
Connecting to image-net.org (image-net.org)|171.64.68.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6744924160 (6.3G) [application/x-tar]
Saving to: ‘ILSVRC2012_img_val.tar’

ILSVRC2012_img_val.  43%[=======>            ]   2.72G  3.86MB/s    eta 16m 23s

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



ILSVRC2012_img_val. 100%[===================>]   6.28G  4.11MB/s    in 28m 11s 

2026-05-27 20:23:21 (3.80 MB/s) - ‘ILSVRC2012_img_val.tar’ saved [6744924160/6744924160]

--2026-05-27 20:23:22--  https://image-net.org/data/ILSVRC/2012/ILSVRC2012_devkit_t12.tar.gz
Resolving image-net.org (image-net.org)... 171.64.68.16
Connecting to image-net.org (image-net.org)|171.64.68.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2568145 (2.4M) [application/x-gzip]
Saving to: ‘ILSVRC2012_devkit_t12.tar.gz’

ILSVRC2012_devkit_t 100%[===================>]   2.45M   811KB/s    in 3.1s    

2026-05-27 20:23:26 (811 KB/s) - ‘ILSVRC2012_devkit_t12.tar.gz’ saved [2568145/2568145]

Extracting validation images...
Extracting devkit...
Parsing validation ground truth...
Organizing into class folders...
✓ ImageNet validation set organized into 1000 class folders
Total images: 50000
Dataset ready at: ./imagenet_val_organized


In [4]:
# Custom results directory
!torchrun --nnodes=1 --nproc_per_node=1 train_x2_finetune.py \
    --model DiT-XL/2 \
    --data-path ./imagenet_val_organized \
    --results-dir /content/drive/MyDrive/dit_checkpoints \
    --classes 0 1 2 3 4 5 6 7 8 9 \
    --epochs 2 \
    --global-batch-size 50 \
    --log-every 10 \
    --ckpt-every 100


^C
[W528 12:21:04.430689117 TCPStore.cpp:340] [c10d] TCP client failed to connect/validate to host 4090:34609 - retrying (try=0, timeout=300000ms, delay=364ms): Interrupted system call
Exception raised from tryConnect at /pytorch/torch/csrc/distributed/c10d/socket.cpp:925 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >) + 0x9d (0x77cb7236afad in /home/mahmood/github/DuoDiT/.venv/lib/python3.13/site-packages/torch/lib/libc10.so)
frame #1: <unknown function> + 0x1685a01 (0x77cad6285a01 in /home/mahmood/github/DuoDiT/.venv/lib/python3.13/site-packages/torch/lib/libtorch_cpu.so)
frame #2: <unknown function> + 0x69919ef (0x77cadb5919ef in /home/mahmood/github/DuoDiT/.venv/lib/python3.13/site-packages/torch/lib/libtorch_cpu.so)
frame #3: <unknown function> + 0x6991ec2 (0x77cadb591ec2 in /home/mahmood/github/DuoDiT/.venv/lib/python3.13/site-packages/torch/lib/libtorch_cpu.so)
frame #4: <